## **Word2Vec (CBOW)**

Word2Vec is a family of algorithms introduced by Mikolov et al. in 2013 to learn dense vector representations (embeddings) of words from large text corpora. These embeddings capture semantic relationships, such as similarity. There are two main architectures: **Continuous Bag-of-Words (CBOW)** and **Skip-gram**. We'll focus on CBOW, which is efficient for frequent words and predicts a target word given its surrounding context words.

CBOW treats the context as a "bag" (unordered set) of words and aims to maximize the probability of the target word given that context. It's a shallow neural network with no hidden non-linearities beyond the embeddings, trained via backpropagation.


### Python Implementation from Scratch

Below is a complete, self-contained Python implementation of **CBOW** using NumPy (no other ML libraries). It uses full softmax for simplicity, suitable for small corpora/vocabs. I've included comments for clarity. For large datasets, you'd need optimizations like negative sampling (I'll add a note on how to extend it).

### Step 1: Problem Setup and Notation
- **Corpus**: A sequence of words $w_1, w_2, \dots, w_T$, where $T$ is the total number of words.
- **Vocabulary**: A set of unique words $V = \{v_1, v_2, \dots, v_{|V|}\}$, with size $|V|$.
- **Context and Center**: For a center word $w_t$, the context is a window of $c$ words on each side, e.g., $[w_{t-c}, \dots, w_{t-1}, w_{t+1}, \dots, w_{t+c}]$. The context size is $2c$.
- **One-Hot Encoding**: Each word $w$ is represented as a one-hot vector $\mathbf{x} \in \mathbb{R}^{|V|}$, where $x_i = 1$ if $i$ corresponds to $w$, else 0.
- **Embeddings**: We learn two weight matrices:
  - Input embedding matrix $\mathbf{W} \in \mathbb{R}^{N \times |V|}$, where $N$ is the embedding dimension (e.g., 100-300). The embedding for a word with one-hot $\mathbf{x}$ is $\mathbf{h} = \mathbf{W} \mathbf{x} \in \mathbb{R}^N$.
  - Output embedding matrix $\mathbf{W'} \in \mathbb{R}^{|V| \times N}$, used to compute scores for the output layer.
- **Objective**: Maximize the average log probability over the corpus:
  $$J = \frac{1}{T} \sum_{t=1}^T \log P(w_t | w_{t-c}, \dots, w_{t-1}, w_{t+1}, \dots, w_{t+c})$$
  This is equivalent to minimizing the negative log-likelihood (cross-entropy loss).


In [1]:
import numpy as np
from collections import defaultdict, Counter
import random

class CBOW:
    def __init__(self, corpus, vocab_size=None, embedding_dim=100, window_size=2, learning_rate=0.025, epochs=5, min_count=5):
        self.corpus = corpus.lower().split()  # Simple tokenization
        self.embedding_dim = embedding_dim
        self.window_size = window_size
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.min_count = min_count
        self.build_vocab(vocab_size)
        self.initialize_weights()
    
    def build_vocab(self, vocab_size=None):
        # Count word frequencies
        word_counts = Counter(self.corpus)
        # Filter words by min_count
        self.vocab = [word for word, count in word_counts.most_common(vocab_size) if count >= self.min_count]
        self.vocab_size = len(self.vocab)
        self.word_to_idx = {word: idx for idx, word in enumerate(self.vocab)}
        self.idx_to_word = {idx: word for word, idx in self.word_to_idx.items()}
        print(f"Vocabulary size: {self.vocab_size}")
    
    def initialize_weights(self):
        # Input embeddings: N x |V|
        self.W = np.random.uniform(-0.5 / self.embedding_dim, 0.5 / self.embedding_dim, (self.embedding_dim, self.vocab_size))
        # Output embeddings: |V| x N
        self.W_prime = np.random.uniform(-0.5 / self.embedding_dim, 0.5 / self.embedding_dim, (self.vocab_size, self.embedding_dim))
    
    def one_hot(self, idx):
        return np.eye(self.vocab_size)[idx]

### Code Explanation:
This step sets up the data structure and parameters, corresponding to defining the corpus, vocabulary, and embeddings in the mathematical formulation.
- **Initialization:** The `__init__` method sets hyperparameters (embedding dimension $N$, window size $c$, learning rate $\eta$, etc.) and preprocesses the corpus.
- **Vocabulary Building:** `build_vocab` counts word frequencies, filters out rare words (below `min_count`), and creates mappings between words and indices. This defines the vocabulary size $|V|$.
- **Weight Initialization:** `initialize_weights` creates the input embedding matrix $\mathbf{W}$ (shape $N \times |V|$) and output matrix $\mathbf{W'}$ (shape $ |V| \times N $), initialized randomly with small values to avoid saturation.
- **One-Hot Encoding:** `one_hot` generates a _one-hot_ vector for a word index, used for the target word in training.


### Step 2: Forward Pass
The network has three layers: input (context), hidden (average embedding), and output (softmax).

1. **Input Layer**: For each context word $w_{t+j}$ (where $j \in [-c, c] \setminus \{0\}$), get its one-hot vector $\mathbf{x}_{t+j}$.

2. **Hidden Layer**: Compute the average embedding of the context: $$\mathbf{h} = \frac{1}{2c} \sum_{j=-c, j \neq 0}^{c} \mathbf{W} \mathbf{x}_{t+j}$$
   This $\mathbf{h} \in \mathbb{R}^N$ is the projection of the context into the embedding space.

3. **Output Layer**: Compute unnormalized scores (logits) for all words in the vocabulary $\mathbf{u} = \mathbf{W'} \mathbf{h} \in \mathbb{R}^{|V|}$ Then, apply softmax to get probabilities $P(w_o | \text{context}) = y_o = \frac{\exp(u_o)}{\sum_{k=1}^{|V|} \exp(u_k)}$
   where $o$ is the index of the center word $w_t$.

The predicted probability for the true center is $y_t$, and we want this to be close to 1.


In [2]:
def forward(self, context_idxs):
    # Average context embeddings: h = (1/M) * sum(W[:, ctx] for ctx in context)
    h = np.zeros(self.embedding_dim)
    for ctx in context_idxs:
        h += self.W[:, ctx]  # W is N x |V|, so W[:, ctx] is embedding
    h /= len(context_idxs)  # Average
    
    # Logits u = W_prime @ h
    u = self.W_prime @ h
    
    # Softmax y
    y = self.softmax(u)
    return h, u, y

def softmax(self, x):
    exp_x = np.exp(x - np.max(x))  # Numerical stability
    return exp_x / exp_x.sum(axis=0)

### Code Explanation:
This step implements the forward propagation, mapping context words to a probability distribution over the vocabulary for the target word.
- **Hidden Layer:** The forward method takes a list of context word indices (`context_idxs`). For each context word, it retrieves the corresponding column from $\mathbf{W}$ (embedding vector) and computes the average to get $\mathbf{h}$.
- **Output Layer:** Computes `logits` $\mathbf{u} = \mathbf{W'} \mathbf{h}$ using matrix multiplication (NumPy’s @ operator).
- **Softmax:** The softmax method converts logits to probabilities, subtracting the maximum for numerical stability to prevent overflow in $\exp(u_k)$.



### Step 3: Loss Function
For a single example, the loss is the negative log-likelihood (categorical cross-entropy):
$$
E = -\log y_t = -u_t + \log \left( \sum_{k=1}^{|V|} \exp(u_k) \right)
$$
Over a batch or the entire dataset, we average this loss.

In practice, for large $|V|$ (e.g., 10,000+), computing the full softmax is expensive. The original Word2Vec uses approximations like Negative Sampling or Hierarchical Softmax, but for this from-scratch implementation, we'll use full softmax for simplicity (assuming small vocabularies or toy examples). Negative Sampling modifies the loss to:
$$
E = -\log \sigma(u_t) - \sum_{k=1}^K \log \sigma(-u_k)
$$
where $\sigma$ is sigmoid, and $K$ negative samples are drawn from a noise distribution (e.g., unigram^{3/4}). We'll implement the basic version first and note the extension.


In [3]:
def compute_loss(self, u, target):
    # Loss E = -u_target + log(sum exp(u_k))
    return -u[target] + np.log(np.sum(np.exp(u)))

### Code Explanation:
This step quantifies the error between predicted probabilities and the true target, driving the optimization.
- This function (added for clarity, though embedded in train in the original code) computes the cross-entropy loss for a single example.
- It takes the logits $ \mathbf{u} $ and the target word index target, computing $ -u_{\text{target}} + \log(\sum \exp(u_k)) $.
- In the original code, this is part of the training loop (see below), but isolating it here clarifies the loss calculation step.


### Step 4: Backpropagation and Gradients
We update $\mathbf{W}$ and $\mathbf{W'}$ using gradient descent: $\theta \leftarrow \theta - \eta \nabla_\theta E$, where $\eta$ is the learning rate.

1. **Gradient w.r.t. Output Weights $\mathbf{W'}$**:
   The error at the output is $\mathbf{e} = \mathbf{y} - \mathbf{t}$, where $\mathbf{t}$ is the one-hot target (t_t = 1, others 0), and $\mathbf{y}$ is the softmax output.
   $$
   \frac{\partial E}{\partial u_k} = y_k - t_k
   $$
   Then, the gradient for the row of $\mathbf{W'}$ corresponding to word $k$:
   $$
   \frac{\partial E}{\partial \mathbf{w'}_k} = (y_k - t_k) \mathbf{h}^T
   $$
   So, update $\mathbf{W'} \leftarrow \mathbf{W'} - \eta \mathbf{e} \mathbf{h}^T$ (outer product).

2. **Gradient w.r.t. Input Weights $\mathbf{W}$**:
   The backpropagated error to the hidden layer is:
   $$
   \frac{\partial E}{\partial \mathbf{h}} = {\mathbf{W'}}^T \mathbf{e}
   $$
   Since $\mathbf{h}$ is the average of context embeddings, the gradient flows to each context word's column in $\mathbf{W}$:
   $$
   \frac{\partial E}{\partial \mathbf{w}_{t+j}} = \frac{1}{2c} \frac{\partial E}{\partial \mathbf{h}} \quad \forall j \in [-c, c] \setminus \{0\}
   $$
   Update the corresponding columns of $\mathbf{W}$.

We repeat this for each training example (or batch) over multiple epochs.


In [4]:
def backward(self, context_idxs, target, h, y):
    # Error: e = y - t (t is one-hot target)
    t = self.one_hot(target)
    e = y - t
    
    # Update W_prime: W_prime -= lr * e.outer(h)
    dW_prime = np.outer(e, h)
    self.W_prime -= self.learning_rate * dW_prime
    
    # Backprop to h: dh = W_prime.T @ e
    dh = self.W_prime.T @ e
    
    # Update W for each context word
    for ctx in context_idxs:
        dW_ctx = (1.0 / len(context_idxs)) * dh
        self.W[:, ctx] -= self.learning_rate * dW_ctx

### Code Explanation:
This step computes and applies gradients to update the model parameters, aligning with the mathematical derivation. 
- This function (extracted from the `train` loop for clarity) handles backpropagation.
- **Output Weights:** Computes the error $ \mathbf{e} = \mathbf{y} - \mathbf{t} $, then updates $ \mathbf{W'} $ using the gradient $ \mathbf{e} \mathbf{h}^T $ (via `np.outer`).
- **Hidden Layer Error:** Backpropagates the error to $ \mathbf{h} $: $ \frac{\partial E}{\partial \mathbf{h}} = {\mathbf{W'}}^T \mathbf{e} $.
- **Input Weights:** Distributes the gradient to each context word’s embedding, scaled by $ \frac{1}{2c} $ (number of context words), updating the corresponding columns of $ \mathbf{W} $.


### Step 5: Training and Hyperparameters
- **Initialization**: Initialize $\mathbf{W}$ and $\mathbf{W'}$ randomly, e.g., uniform [-0.5/N, 0.5/N] to avoid saturation.
- **Subsampling and Negatives**: For efficiency, subsample frequent words and use negative sampling (sample negatives proportional to frequency^{0.75}).
- **Hyperparameters**: Embedding dim $N$, window size $c$, learning rate $\eta$ (often starts high and decays), epochs, min word count for vocab.
- After training, the input embeddings $\mathbf{W}$ are typically used as the final word vectors.

This setup learns embeddings where similar words have close vectors (via dot product similarity).


In [5]:
def generate_training_data(self):
    training_data = []
    for i in range(len(self.corpus)):
        word = self.corpus[i]
        if word not in self.word_to_idx:
            continue  # Skip OOV
        target_idx = self.word_to_idx[word]
        context = []
        for j in range(-self.window_size, self.window_size + 1):
            if j == 0:
                continue
            if 0 <= i + j < len(self.corpus):
                ctx_word = self.corpus[i + j]
                if ctx_word in self.word_to_idx:
                    context.append(self.word_to_idx[ctx_word])
        if context:
            training_data.append((context, target_idx))
    return training_data

def train(self):
    training_data = self.generate_training_data()
    for epoch in range(self.epochs):
        total_loss = 0
        for context, target in training_data:
            h, u, y = self.forward(context)
            
            # Compute loss
            E = self.compute_loss(u, target)
            total_loss += E
            
            # Backprop
            self.backward(context, target, h, y)
        
        print(f"Epoch {epoch + 1}/{self.epochs}, Loss: {total_loss / len(training_data):.4f}")
    
def get_embedding(self, word):
    if word in self.word_to_idx:
        return self.W[:, self.word_to_idx[word]]
    return None

### Code Explanation:
This step ties everything together, iterating over the dataset and optimizing the embeddings using the computed gradients.
- **Data Generation:** generate_training_data creates training pairs (context indices, target index) by sliding a window of size $ 2c $ over the corpus, skipping out-of-vocabulary words.
- **Training Loop:** The train method iterates over epochs, processing each training example:

    * Calls forward to get $ \mathbf{h} $, $ \mathbf{u} $, and $ \mathbf{y} $.
    * Computes the loss (via compute_loss, inlined in the original code).
    * Calls backward to update $ \mathbf{W} $ and $ \mathbf{W'} $.
    * Reports the average loss per epoch.


- **Embedding Retrieval:** `get_embedding` extracts the learned embedding for a word from $ \mathbf{W} $.


### Complete Code with Steps Integrated
Below is the complete code with the steps clearly separated, incorporating the isolated compute_loss and backward functions for clarity. It’s equivalent to the original but modularized to match the five steps.

In [ ]:
import numpy as np
from collections import defaultdict, Counter
import random
import uuid

class CBOW:
    # Step 1: Problem Setup (Updated for Negative Sampling)
    def __init__(self, corpus, vocab_size=None, embedding_dim=100, window_size=2, learning_rate=0.025, epochs=5, min_count=5, neg_samples=5):
        self.corpus = corpus.lower().split()
        self.embedding_dim = embedding_dim
        self.window_size = window_size
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.min_count = min_count
        self.neg_samples = neg_samples  # Number of negative samples
        self.build_vocab(vocab_size)
        self.initialize_weights()
        self.build_unigram_table()
    
    def build_vocab(self, vocab_size=None):
        word_counts = Counter(self.corpus)
        self.vocab = [word for word, count in word_counts.most_common(vocab_size) if count >= self.min_count]
        self.vocab_size = len(self.vocab)
        self.word_to_idx = {word: idx for idx, word in enumerate(self.vocab)}
        self.idx_to_word = {idx: word for word, idx in self.word_to_idx.items()}
        # Store word frequencies for negative sampling
        self.word_freqs = np.array([word_counts[word] for word in self.vocab], dtype=float)
        self.word_freqs = self.word_freqs ** 0.75  # Unigram^0.75
        self.word_freqs /= self.word_freqs.sum()  # Normalize to probabilities
        print(f"Vocabulary size: {self.vocab_size}")
    
    def build_unigram_table(self):
        # Create a table for efficient negative sampling
        table_size = 1_000_000
        self.unigram_table = []
        for idx in range(self.vocab_size):
            count = int(self.word_freqs[idx] * table_size)
            self.unigram_table.extend([idx] * count)
    
    def initialize_weights(self):
        self.W = np.random.uniform(-0.5 / self.embedding_dim, 0.5 / self.embedding_dim, (self.embedding_dim, self.vocab_size))
        self.W_prime = np.random.uniform(-0.5 / self.embedding_dim, 0.5 / self.embedding_dim, (self.vocab_size, self.embedding_dim))
    
    def one_hot(self, idx):
        return np.eye(self.vocab_size)[idx]
    
    # Step 2: Forward Pass (Adjusted for Negative Sampling)
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))
    
    def forward(self, context_idxs):
        h = np.zeros(self.embedding_dim)
        for ctx in context_idxs:
            h += self.W[:, ctx]
        h /= len(context_idxs)
        return h
    
    # Step 3: Loss Function (Negative Sampling)
    def compute_loss(self, h, target, neg_samples):
        # Positive sample
        u_pos = np.dot(self.W_prime[target], h)
        loss = -np.log(self.sigmoid(u_pos))
        
        # Negative samples
        for neg_idx in neg_samples:
            u_neg = np.dot(self.W_prime[neg_idx], h)
            loss -= np.log(self.sigmoid(-u_neg))
        
        return loss
    
    # Step 4: Backpropagation and Gradients (Negative Sampling)
    def backward(self, context_idxs, target, h, neg_samples):
        # Gradient for positive sample
        u_pos = np.dot(self.W_prime[target], h)
        grad_pos = self.sigmoid(u_pos) - 1
        dh = grad_pos * self.W_prime[target]
        self.W_prime[target] -= self.learning_rate * grad_pos * h
        
        # Gradients for negative samples
        for neg_idx in neg_samples:
            u_neg = np.dot(self.W_prime[neg_idx], h)
            grad_neg = self.sigmoid(u_neg)
            dh += grad_neg * self.W_prime[neg_idx]
            self.W_prime[neg_idx] -= self.learning_rate * grad_neg * h
        
        # Update input embeddings
        for ctx in context_idxs:
            dW_ctx = (1.0 / len(context_idxs)) * dh
            self.W[:, ctx] -= self.learning_rate * dW_ctx
    
    # Step 5: Training and Hyperparameters
    def generate_training_data(self):
        training_data = []
        for i in range(len(self.corpus)):
            word = self.corpus[i]
            if word not in self.word_to_idx:
                continue
            target_idx = self.word_to_idx[word]
            context = []
            for j in range(-self.window_size, self.window_size + 1):
                if j == 0:
                    continue
                if 0 <= i + j < len(self.corpus):
                    ctx_word = self.corpus[i + j]
                    if ctx_word in self.word_to_idx:
                        context.append(self.word_to_idx[ctx_word])
            if context:
                training_data.append((context, target_idx))
        return training_data
    
    def sample_negatives(self, target):
        # Sample K negative words, excluding the target
        negatives = []
        while len(negatives) < self.neg_samples:
            idx = random.choice(self.unigram_table)
            if idx != target:
                negatives.append(idx)
        return negatives
    
    def train(self):
        training_data = self.generate_training_data()
        for epoch in range(self.epochs):
            total_loss = 0
            for context, target in training_data:
                h = self.forward(context)
                neg_samples = self.sample_negatives(target)
                E = self.compute_loss(h, target, neg_samples)
                total_loss += E
                self.backward(context, target, h, neg_samples)
            print(f"Epoch {epoch + 1}/{self.epochs}, Loss: {total_loss / len(training_data):.4f}")
    
    def get_embedding(self, word):
        if word in self.word_to_idx:
            return self.W[:, self.word_to_idx[word]]
        return None


### Example usage

In [9]:

corpus = "We are about to study the idea of a computational process. Computational processes are abstract beings that inhabit computers. As they evolve, processes manipulate other abstract things called data. The evolution of a process is directed by a pattern of rules called a program. People create programs to direct processes. In effect, we conjure the spirits of the computer with our spells."
model = CBOW(corpus, embedding_dim=10, window_size=2, epochs=10, min_count=1)
model.train()

# Get embedding for a word
print("Embedding for 'process':", model.get_embedding('process'))

Vocabulary size: 46
Epoch 1/10, Loss: 4.1585
Epoch 2/10, Loss: 4.1587
Epoch 3/10, Loss: 4.1585
Epoch 4/10, Loss: 4.1584
Epoch 5/10, Loss: 4.1586
Epoch 6/10, Loss: 4.1585
Epoch 7/10, Loss: 4.1583
Epoch 8/10, Loss: 4.1584
Epoch 9/10, Loss: 4.1585
Epoch 10/10, Loss: 4.1579
Embedding for 'process': [-0.03540107 -0.03837333 -0.00614167 -0.02597846  0.04521524  0.02097014
 -0.01957327 -0.03521638  0.01382244  0.03641831]


This breakdown makes the implementation modular and directly traceable to the mathematical steps, aiding understanding. For large-scale use, consider adding negative sampling or hierarchical softmax to reduce computational cost, as noted in the original explanation. You can run this code with a small corpus as shown, and it will produce embeddings for words like "process".

---

## **Negative Sampling**
Negative Sampling is an optimization technique used in training models like Word2Vec (including CBOW and Skip-gram) to make the learning process more computationally efficient, especially for large vocabularies. Below, We will explain negative sampling in the context of Word2Vec’s CBOW model, focusing on its purpose, mechanics, and mathematical foundation.


To update the CBOW implementation with **negative sampling**, we need to modify the loss function and gradient computation to make training more efficient, especially for large vocabularies. Negative sampling avoids computing the full softmax by approximating the loss using the target word and a small number of randomly sampled "negative" words. Below, we’ll explain the changes in the context of the mathematical background and provide the updated code, keeping the same five-step structure from the previous steps. The changes primarily affect **Step 3 (Loss Function)** and **Step 4 (Backpropagation)**, with minor adjustments elsewhere to support negative sampling.

### Mathematical Background for Negative Sampling
In the original CBOW implementation, we used the full softmax loss:
$$
E = -\log P(w_t | \text{context}) = -u_t + \log \left( \sum_{k=1}^{|V|} \exp(u_k) \right)
$$
This requires computing $ \exp(u_k) $ for all words in the vocabulary $|V|$, which is computationally expensive for large vocabularies (e.g., $|V| > 10,000$).

**Negative Sampling** approximates the loss by considering only the target word (positive example) and $K$ randomly sampled negative words (typically $K = 5-20 $). The loss becomes: $E = -\log \sigma(u_t) - \overset{K}{\underset{k=1}{\sum}} \log \sigma(-u_k)$

where:
- $ \sigma(x) = \frac{1}{1 + \exp(-x)}$ is the sigmoid function.
- $ u_t = \mathbf{w'}_t \cdot \mathbf{h}$ is the score for the target word.
- $ u_k = \mathbf{w'}_k \cdot \mathbf{h}$ is the score for a negative word $ k $.
- Negative words are sampled from a noise distribution, often the unigram distribution raised to the power of 0.75: $P(w) \propto \text{freq}(w)^{0.75}$.


Negative Sampling reduces the computational cost from $O(|V|)$ to $O(K + 1)$ per training example. and the gradients only update the weights for the target word and the $K$ negative words, not the entire $\mathbf{W'}$.

**Gradient Updates**:
- For the target word $t$:$\frac{\partial E}{\partial u_t} = \sigma(u_t) - 1$

  Update $\mathbf{w'}_t \leftarrow \mathbf{w'}_t - \eta (\sigma(u_t) - 1) \mathbf{h}$.
- For a negative word $k$: $\frac{\partial E}{\partial u_k} = \sigma(u_k)$

  Update $\mathbf{w'}_k \leftarrow \mathbf{w'}_k - \eta \sigma(u_k) \mathbf{h}$.
- The error backpropagated to the hidden layer is: $\frac{\partial E}{\partial \mathbf{h}} = (\sigma(u_t) - 1) \mathbf{w'}_t + \sum_{k=1}^K \sigma(u_k) \mathbf{w'}_k$

- The input embeddings $\mathbf{W}$ are updated as before, distributing $\frac{\partial E}{\partial \mathbf{h}}$ to each context word.

**Sampling Negative Words**:
- We sample negatives from $P(w) \propto \text{freq}(w)^{0.75}$, which gives more weight to frequent words but less than their raw frequency, balancing informativeness.

### Updated Code with Negative Sampling
The code retains the five-step structure, with changes in **Step 3 (Loss Function)** and **Step 4 (Backpropagation)** to implement negative sampling. Additionally, we modify **Step 1 (Problem Setup)** to compute the unigram distribution for sampling negatives. The forward pass and training loop are adjusted slightly to accommodate the new loss and gradient updates.

In [10]:
import numpy as np
from collections import defaultdict, Counter
import random
import uuid

class CBOW:
    # Step 1: Problem Setup (Updated for Negative Sampling)
    def __init__(self, corpus, vocab_size=None, embedding_dim=100, window_size=2, learning_rate=0.025, epochs=5, min_count=5, neg_samples=5):
        self.corpus = corpus.lower().split()
        self.embedding_dim = embedding_dim
        self.window_size = window_size
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.min_count = min_count
        self.neg_samples = neg_samples  # Number of negative samples
        self.build_vocab(vocab_size)
        self.initialize_weights()
        self.build_unigram_table()
    
    def build_vocab(self, vocab_size=None):
        word_counts = Counter(self.corpus)
        self.vocab = [word for word, count in word_counts.most_common(vocab_size) if count >= self.min_count]
        self.vocab_size = len(self.vocab)
        self.word_to_idx = {word: idx for idx, word in enumerate(self.vocab)}
        self.idx_to_word = {idx: word for word, idx in self.word_to_idx.items()}
        # Store word frequencies for negative sampling
        self.word_freqs = np.array([word_counts[word] for word in self.vocab], dtype=float)
        self.word_freqs = self.word_freqs ** 0.75  # Unigram^0.75
        self.word_freqs /= self.word_freqs.sum()  # Normalize to probabilities
        print(f"Vocabulary size: {self.vocab_size}")
    
    def build_unigram_table(self):
        # Create a table for efficient negative sampling
        table_size = 1_000_000
        self.unigram_table = []
        for idx in range(self.vocab_size):
            count = int(self.word_freqs[idx] * table_size)
            self.unigram_table.extend([idx] * count)
    
    def initialize_weights(self):
        self.W = np.random.uniform(-0.5 / self.embedding_dim, 0.5 / self.embedding_dim, (self.embedding_dim, self.vocab_size))
        self.W_prime = np.random.uniform(-0.5 / self.embedding_dim, 0.5 / self.embedding_dim, (self.vocab_size, self.embedding_dim))
    
    def one_hot(self, idx):
        return np.eye(self.vocab_size)[idx]
    
    # Step 2: Forward Pass (Adjusted for Negative Sampling)
    def sigmoid(self, x):
        return 1 / (1 + np.exp(-x))
    
    def forward(self, context_idxs):
        h = np.zeros(self.embedding_dim)
        for ctx in context_idxs:
            h += self.W[:, ctx]
        h /= len(context_idxs)
        return h
    
    # Step 3: Loss Function (Negative Sampling)
    def compute_loss(self, h, target, neg_samples):
        # Positive sample
        u_pos = np.dot(self.W_prime[target], h)
        loss = -np.log(self.sigmoid(u_pos))
        
        # Negative samples
        for neg_idx in neg_samples:
            u_neg = np.dot(self.W_prime[neg_idx], h)
            loss -= np.log(self.sigmoid(-u_neg))
        
        return loss
    
    # Step 4: Backpropagation and Gradients (Negative Sampling)
    def backward(self, context_idxs, target, h, neg_samples):
        # Gradient for positive sample
        u_pos = np.dot(self.W_prime[target], h)
        grad_pos = self.sigmoid(u_pos) - 1
        dh = grad_pos * self.W_prime[target]
        self.W_prime[target] -= self.learning_rate * grad_pos * h
        
        # Gradients for negative samples
        for neg_idx in neg_samples:
            u_neg = np.dot(self.W_prime[neg_idx], h)
            grad_neg = self.sigmoid(u_neg)
            dh += grad_neg * self.W_prime[neg_idx]
            self.W_prime[neg_idx] -= self.learning_rate * grad_neg * h
        
        # Update input embeddings
        for ctx in context_idxs:
            dW_ctx = (1.0 / len(context_idxs)) * dh
            self.W[:, ctx] -= self.learning_rate * dW_ctx
    
    # Step 5: Training and Hyperparameters
    def generate_training_data(self):
        training_data = []
        for i in range(len(self.corpus)):
            word = self.corpus[i]
            if word not in self.word_to_idx:
                continue
            target_idx = self.word_to_idx[word]
            context = []
            for j in range(-self.window_size, self.window_size + 1):
                if j == 0:
                    continue
                if 0 <= i + j < len(self.corpus):
                    ctx_word = self.corpus[i + j]
                    if ctx_word in self.word_to_idx:
                        context.append(self.word_to_idx[ctx_word])
            if context:
                training_data.append((context, target_idx))
        return training_data
    
    def sample_negatives(self, target):
        # Sample K negative words, excluding the target
        negatives = []
        while len(negatives) < self.neg_samples:
            idx = random.choice(self.unigram_table)
            if idx != target:
                negatives.append(idx)
        return negatives
    
    def train(self):
        training_data = self.generate_training_data()
        for epoch in range(self.epochs):
            total_loss = 0
            for context, target in training_data:
                h = self.forward(context)
                neg_samples = self.sample_negatives(target)
                E = self.compute_loss(h, target, neg_samples)
                total_loss += E
                self.backward(context, target, h, neg_samples)
            print(f"Epoch {epoch + 1}/{self.epochs}, Loss: {total_loss / len(training_data):.4f}")
    
    def get_embedding(self, word):
        if word in self.word_to_idx:
            return self.W[:, self.word_to_idx[word]]
        return None

In [11]:
# Example usage
corpus = "We are about to study the idea of a computational process. Computational processes are abstract beings that inhabit computers. As they evolve, processes manipulate other abstract things called data. The evolution of a process is directed by a pattern of rules called a program. People create programs to direct processes. In effect, we conjure the spirits of the computer with our spells."
model = CBOW(corpus, embedding_dim=10, window_size=2, epochs=10, min_count=1, neg_samples=5)
model.train()
print("Embedding for 'process':", model.get_embedding('process'))

Vocabulary size: 46
Epoch 1/10, Loss: 4.1589
Epoch 2/10, Loss: 4.1587
Epoch 3/10, Loss: 4.1584
Epoch 4/10, Loss: 4.1583
Epoch 5/10, Loss: 4.1583
Epoch 6/10, Loss: 4.1580
Epoch 7/10, Loss: 4.1583
Epoch 8/10, Loss: 4.1577
Epoch 9/10, Loss: 4.1576
Epoch 10/10, Loss: 4.1571
Embedding for 'process': [ 0.00427353 -0.01042316  0.00385249  0.02805783 -0.0520574   0.00324293
  0.04986753 -0.01191904  0.01627711 -0.007118  ]
